# Silver Agent Implementation

Reads unmerged bronze agents, cleans and validates them, MERGEs valid rows into silver,
quarantines bad rows, and flags processed bronze rows.

| Outcome | Destination | Bronze `merge_flag` |
|---|---|---|
| Valid | `silver.agent` | `true` |
| Hard reject (bad data) | `silver.quarantine_agent` | `true` |
| Soft reject: branch not (yet) in `silver.branch` | stays in bronze | `false` (retried next run) |

**Job order:** depends on Silver_Branch.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.agent (
  agent_id         INT,
  agent_name       STRING,
  agent_email      STRING,
  agent_phone      STRING,
  branch_id        INT,
  create_timestamp TIMESTAMP,
  merged_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/agent';

CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.quarantine_agent (
  agent_id              INT,
  agent_name            STRING,
  agent_email           STRING,
  agent_phone           STRING,
  branch_id             INT,
  create_timestamp      TIMESTAMP,
  reject_reason         STRING,
  quarantined_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/quarantine/agent';

In [0]:
from pyspark.sql import functions as F, Window

COLUMNS = ["agent_id", "agent_name", "agent_email", "agent_phone", "branch_id", "create_timestamp"]
SOFT_REASONS = ["missing_branch"]  # parent may arrive later -> retry instead of quarantine

# 1. Read only unmerged bronze agents
agent_df = spark.sql("SELECT * FROM policysystemdatabricks.bronze.agent WHERE merge_flag = false")

# 2. Clean: trim / collapse name spaces, lower-case email, default missing values
clean_df = agent_df.selectExpr(
    "agent_id",
    r"coalesce(nullif(regexp_replace(trim(agent_name), '\\s+', ' '), ''), 'Unknown Agent') AS agent_name",
    "coalesce(nullif(lower(trim(agent_email)), ''), 'admin@azurelib.com') AS agent_email",
    "coalesce(nullif(trim(agent_phone), ''), '0000000000') AS agent_phone",
    "branch_id",
    "create_timestamp",
)

# 3. Deduplicate: one row per agent_id (latest create_timestamp). Extra sort keys make the pick deterministic.
#    NULL agent_id rows are kept individually so each one is quarantined.
w = Window.partitionBy("agent_id").orderBy(
    F.col("create_timestamp").desc_nulls_last(), F.col("agent_email"), F.col("agent_phone"))
df_result = (clean_df.withColumn("rn", F.row_number().over(w))
             .filter("rn = 1 OR agent_id IS NULL")
             .drop("rn"))

In [0]:
# Referential check against SILVER branches, done here so missing branches are counted, not silently dropped.
silver_branches = (spark.table("policysystemdatabricks.silver.branch")
                   .select("branch_id").distinct()
                   .withColumn("_branch_exists", F.lit(True)))

reject_reason = (
    F.when(F.col("agent_id").isNull(), "null_agent_id")
     .when(~F.col("agent_phone").rlike(r"^[0-9]{10}$"), "invalid_phone")
     .when(~F.col("agent_email").rlike(r"^[^@ ]+@[^@ ]+\.[^@ ]+$"), "invalid_email")
     .when(F.col("branch_id").isNull(), "null_branch_id")
     .when(F.col("_branch_exists").isNull(), "missing_branch")
)

classified_df = (df_result
                 .join(silver_branches, on="branch_id", how="left")
                 .withColumn("reject_reason", reject_reason)
                 .select(*COLUMNS, "reject_reason"))
classified_df.createOrReplaceTempView("agent_classified")

display(classified_df.groupBy(F.coalesce("reject_reason", F.lit("VALID")).alias("outcome")).count())

outcome,count


In [0]:
display(spark.sql("""
MERGE INTO policysystemdatabricks.silver.agent AS t
USING (SELECT * FROM agent_classified WHERE reject_reason IS NULL) AS s
ON t.agent_id = s.agent_id
WHEN MATCHED THEN UPDATE SET
  t.agent_name       = s.agent_name,
  t.agent_email      = s.agent_email,
  t.agent_phone      = s.agent_phone,
  t.branch_id        = s.branch_id,
  t.create_timestamp = s.create_timestamp,
  t.merged_timestamp = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (agent_id, agent_name, agent_email, agent_phone, branch_id, create_timestamp, merged_timestamp)
  VALUES (s.agent_id, s.agent_name, s.agent_email, s.agent_phone, s.branch_id, s.create_timestamp, current_timestamp())
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
soft_list = ", ".join(f"'{r}'" for r in SOFT_REASONS)

display(spark.sql(f"""
INSERT INTO policysystemdatabricks.silver.quarantine_agent
SELECT {", ".join(COLUMNS)}, reject_reason, current_timestamp()
FROM agent_classified
WHERE reject_reason IS NOT NULL AND reject_reason NOT IN ({soft_list})
"""))

num_affected_rows,num_inserted_rows
0,0


In [0]:
# The timestamp guard avoids flagging a newer bronze version that landed after this run read the table.
display(spark.sql(f"""
MERGE INTO policysystemdatabricks.bronze.agent AS b
USING (SELECT agent_id, create_timestamp FROM agent_classified
       WHERE agent_id IS NOT NULL
         AND (reject_reason IS NULL OR reject_reason NOT IN ({soft_list}))) AS s
ON  b.agent_id = s.agent_id
AND b.merge_flag = false
AND (s.create_timestamp IS NULL OR b.create_timestamp IS NULL OR b.create_timestamp <= s.create_timestamp)
WHEN MATCHED THEN UPDATE SET b.merge_flag = true
"""))

# NULL keys cannot be matched in a MERGE; they are always quarantined above, so flag them directly.
display(spark.sql("""
UPDATE policysystemdatabricks.bronze.agent
SET merge_flag = true
WHERE merge_flag = false AND agent_id IS NULL
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


num_affected_rows
0


In [0]:
%sql
SELECT (SELECT count(*) FROM policysystemdatabricks.bronze.agent WHERE merge_flag = false) AS bronze_waiting_for_branch,
       (SELECT count(*) FROM policysystemdatabricks.silver.quarantine_agent) AS quarantined_rows_total,
       count(*)                                  AS silver_rows,
       count(DISTINCT agent_id)                  AS silver_distinct_agents,
       count_if(agent_email = 'admin@azurelib.com') AS default_emails,
       count_if(merged_timestamp IS NULL)        AS rows_missing_merged_timestamp,  -- should be 0
       (SELECT count(*) FROM policysystemdatabricks.silver.agent a
        LEFT ANTI JOIN policysystemdatabricks.silver.branch b ON a.branch_id = b.branch_id) AS agents_without_silver_branch,  -- should be 0
       max(merged_timestamp)                     AS last_merged_at
FROM policysystemdatabricks.silver.agent

bronze_waiting_for_branch,quarantined_rows_total,silver_rows,silver_distinct_agents,default_emails,rows_missing_merged_timestamp,agents_without_silver_branch,last_merged_at
0,0,999,999,59,0,0,2026-10-26T08:39:15.000Z
